# 03 · De páginas sueltas a un corpus

**40 minutos.**

Al terminar vas a tener un archivo con varios artículos, cada uno con su texto y
con los datos que dicen de dónde salió.

Eso es un corpus.

> **Celda de reanudación.** Si recién llegás o se te reinició el kernel, corré
> esta celda: deja todo listo para seguir desde acá.

In [ ]:
import time
from pathlib import Path

import requests

CABECERAS = {
    "User-Agent": "LabPLN 2026 - materia de PLN (tu-mail@ejemplo.com)"
}

DATOS = Path("datos")
DATOS.mkdir(exist_ok=True)


def pedir(url, pausa=1.0):
    """Descarga una página portándose bien."""
    respuesta = requests.get(url, headers=CABECERAS, timeout=20)
    respuesta.raise_for_status()
    time.sleep(pausa)
    return respuesta.text


print("Listo.")

## Una herramienta que ya sabe qué es un artículo

`trafilatura` fue hecha para una sola cosa: dado el HTML de una nota, devolver el
cuerpo de la nota. Descarta menú, publicidad, notas relacionadas y pie de página.

No es magia: son reglas afinadas contra miles de sitios reales. Eso es
exactamente el trabajo que decidimos no hacer a mano.

Fijate que le pasamos el HTML que consiguió **nuestra** función `pedir()`.
Trafilatura trae su propio descargador, pero así mantenemos una sola forma de
pedir páginas en toda la materia: con nuestras cabeceras y nuestra pausa.

In [ ]:
import trafilatura

URL = "https://cenital.com/el-cerebro-tal-vez-no-sea-una-computadora/"

html = pedir(URL)
texto = trafilatura.extract(html)

print(texto[:600])

### Cuánto ruido nos sacó

In [ ]:
from bs4 import BeautifulSoup

sopa = BeautifulSoup(html, "html.parser")

print("BeautifulSoup get_text():", len(sopa.get_text()), "caracteres")
print("trafilatura extract():   ", len(texto), "caracteres")

La diferencia son menús, pies, botones de compartir y notas relacionadas. En un
corpus todo eso es ruido que después vas a contar como si fuera lenguaje del
artículo.

---

## Un corpus es texto **más** procedencia

Un archivo con textos sueltos no es un corpus. Es una carpeta.

Un corpus permite responder: ¿de dónde salió esto?, ¿quién lo escribió?, ¿cuándo
se publicó?, ¿cuándo lo capturé yo? Sin esas respuestas no podés citar, no podés
comparar por fecha, y no podés reproducir tu propio trabajo dentro de seis meses.

Lo bueno: trafilatura ya extrae todo eso. Solo hay que pedírselo.

Le pasamos también la `url`: trafilatura recibe el HTML suelto y no tiene manera
de adivinar de dónde salió. El dato lo tenemos nosotros, que lo descargamos.

In [ ]:
import json

crudo = trafilatura.extract(
    html, url=URL, output_format="json", with_metadata=True
)
datos = json.loads(crudo)

print("Título: ", datos["title"])
print("Autoría:", datos["author"])
print("Fecha:  ", datos["date"])
print("Medio:  ", datos["hostname"])

## El registro de un artículo

Esta función hace todo el trabajo de capturar **un** artículo. Devuelve dos cosas:
el registro y un estado.

In [ ]:
from datetime import date


def capturar(url):
    """Devuelve (registro, estado). Si algo falla, el registro es None."""
    try:
        html = pedir(url)
    except Exception as error:
        return None, f"no se pudo descargar: {error}"

    crudo = trafilatura.extract(
        html, url=url, output_format="json", with_metadata=True
    )
    if crudo is None:
        return None, "no se encontró texto de artículo"

    datos = json.loads(crudo)
    registro = {
        "url": url,
        "medio": datos["hostname"],
        "titulo": datos["title"],
        "autoria": datos["author"],
        "fecha_publicacion": datos["date"],
        "fecha_captura": date.today().isoformat(),
        "texto": datos["text"],
    }
    return registro, "ok"

### Por qué devolvemos `None` y no un mensaje

Es tentador escribir `return "No se pudo extraer el texto"` cuando algo falla.

Es una trampa. Ese mensaje entra en la columna de texto y a partir de ahí es
indistinguible de un artículo. Cuando cuentes palabras, "no", "se" y "pudo" van a
aparecer entre las más frecuentes de tu corpus, y no vas a entender por qué.

**Un error nunca se guarda como si fuera un dato.** Va a otro lado.

## Ahora en lote

Una de estas URLs está rota a propósito.

In [ ]:
URLS = [
    "https://cenital.com/el-cerebro-tal-vez-no-sea-una-computadora/",
    "https://cenital.com/la-fragilidad-de-internet/",
    "https://cenital.com/epstein-y-la-decadencia-del-sistema-cientifico/",
    "https://cenital.com/esta-nota-no-existe-a-proposito/",
]

corpus = []
fallas = []

for url in URLS:
    registro, estado = capturar(url)
    if estado == "ok":
        corpus.append(registro)
        print("OK     ", registro["titulo"][:55])
    else:
        fallas.append({"url": url, "motivo": estado})
        print("FALLÓ  ", url)
        print("        →", estado)

print()
print("Corpus:", len(corpus), "artículos.  Fallas:", len(fallas))

> Si la URL rota **no** falló, es porque el sitio devuelve una página de "no
> encontrado" con código 200 en vez de 404. Pasa, y es justamente por eso que
> siempre hay que mirar lo que se guardó, no solo el resumen.

---

## Guardar: un artículo por línea

Podríamos usar CSV, pero los textos largos tienen comas, comillas y saltos de
línea adentro, y el CSV se rompe justo cuando el corpus se pone interesante.

**JSONL** es un archivo de texto con un registro por línea. No hay que escapar
nada, se puede ir agregando de a uno, y pandas lo lee en una línea.

In [ ]:
ARCHIVO_CORPUS = DATOS / "corpus.jsonl"

with open(ARCHIVO_CORPUS, "w", encoding="utf-8") as archivo:
    for registro in corpus:
        archivo.write(json.dumps(registro, ensure_ascii=False) + "\n")

print("Guardado en:", ARCHIVO_CORPUS)

> `ensure_ascii=False`: sin eso, los acentos se guardan como `\u00e1` y el archivo
> deja de ser legible para un humano. `encoding="utf-8"`: igual que siempre.

In [ ]:
import pandas as pd

df = pd.read_json(ARCHIVO_CORPUS, lines=True)
df[["medio", "fecha_publicacion", "autoria", "titulo"]]

### El CSV sigue sirviendo, pero para otra cosa

Para **compartir** el corpus con alguien que lo va a abrir en una planilla,
exportamos el índice: todos los campos menos los textos.

Así el CSV es liviano, se abre sin romperse, y el corpus de verdad sigue estando
en el JSONL.

In [ ]:
df.drop(columns=["texto"]).to_csv(DATOS / "corpus_indice.csv", index=False)
print("Índice exportado a datos/corpus_indice.csv")

---

## Dos limpiezas distintas (y no hay que mezclarlas)

|  | Limpieza de captura | Preprocesamiento de análisis |
|---|---|---|
| Qué hace | Normaliza espacios y saltos | Minúsculas, saca puntuación y palabras vacías |
| Cuándo | Una vez, al capturar | Cada vez que analizás |
| ¿Se guarda? | **Sí**, es el corpus | **No**, se recalcula |
| Cómo es | Conservadora, reversible | Destructiva, descartable |

La regla: **el corpus se guarda lo más parecido posible al original.** Todo lo que
destruye información se hace después, y así se puede volver a hacer distinto.

In [ ]:
import re


def normalizar(texto):
    """Limpieza de captura: ordena espacios. No borra información."""
    texto = texto.replace("\xa0", " ")       # espacios "duros" del HTML
    texto = re.sub(r"[ \t]+", " ", texto)    # espacios repetidos
    texto = re.sub(r"\n{3,}", "\n\n", texto) # más de dos saltos seguidos
    return texto.strip()


print(repr(normalizar("Hola\xa0\xa0mundo   con    espacios.")))

In [ ]:
STOPWORDS = set(
    (DATOS / "stopwords_es.txt").read_text(encoding="utf-8").split()
)


def palabras_significativas(texto):
    """Preprocesamiento de análisis: destructivo. El resultado NO se guarda."""
    texto = texto.lower()
    texto = re.sub(r"[^\w\s]", " ", texto)
    return [
        palabra
        for palabra in texto.split()
        if palabra not in STOPWORDS and len(palabra) > 2
    ]


print("Palabras vacías cargadas:", len(STOPWORDS))

> `datos/stopwords_es.txt` es un archivo de texto con una palabra por línea.
> **Abrilo.** No hay nada mágico adentro. Si tu análisis necesita sacar otras
> palabras, agregalas ahí — pero anotá por qué las sacaste. Esa decisión es parte
> del análisis, no un detalle técnico.

In [ ]:
from collections import Counter

todo_el_corpus = " ".join(df["texto"])
frecuencias = Counter(palabras_significativas(todo_el_corpus))

for palabra, veces in frecuencias.most_common(15):
    print(f"{veces:4}  {palabra}")

### Y lo mismo, mirado

In [ ]:
import matplotlib.pyplot as plt

top = frecuencias.most_common(15)
palabras = [palabra for palabra, _ in top][::-1]
veces = [cantidad for _, cantidad in top][::-1]

plt.figure(figsize=(8, 5))
plt.barh(palabras, veces)
plt.title("Palabras más frecuentes del corpus")
plt.xlabel("apariciones")
plt.tight_layout()
plt.show()

---

## Lo que hiciste hoy

1. Pediste una página portándote bien: identificándote, esperando, mirando el
   `robots.txt`.
2. Leíste su estructura con BeautifulSoup y aprendiste dónde deja de alcanzar.
3. Extrajiste artículos con su procedencia y los guardaste como corpus.

Ese es el ciclo completo. Todo lo que sigue en la materia entra por acá.